# Assignment: 1
# Vaishnavi B, 27PGAI0120

In [105]:
import json
import os
import re
import time

import pandas as pd
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List, Literal

In [106]:
## Load environment variables and API keys
load_dotenv(override=True)

if "GROQ_API_KEY" not in os.environ:
    raise ValueError("GROQ_API_KEY not found. Add it to the .env file at the project root.")

print(f"Groq API Key exists and begins {os.environ['GROQ_API_KEY'][:4]}")

Groq API Key exists and begins gsk_


### Chat Model

The required parts (30 articles + 25 job postings) run on Groq. A low temperature is used
throughout: classification and extraction are deterministic tasks, so creativity only hurts here.

Two settings matter for this model. `reasoning_format="hidden"` keeps gpt-oss's chain-of-thought
out of the response, and every structured chain below uses `method="json_schema"` rather than the
default tool-calling path - with tool calling, gpt-oss tends to answer in prose ("So the answer is
Business") and the parse fails.

In [107]:
llm = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0,
    reasoning_format="hidden",  ## gpt-oss is a reasoning model; keep the chain-of-thought out of the answer
)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001FBD7890C50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FBD78925D0>, model_name='openai/gpt-oss-120b', temperature=1e-08, reasoning_format='hidden', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [108]:
def retry_after_seconds(exc, default):
    """Seconds Groq asked us to wait, or `default` if the error does not say."""
    match = re.search(r"try again in (?:(\d+)m)?([\d.]+)s", str(exc))
    if not match:
        return default
    return int(match.group(1) or 0) * 60 + float(match.group(2)) + 1


def invoke_with_retry(chain, payload, retries=5, wait=8, max_wait=420):
    for attempt in range(retries):
        try:
            return chain.invoke(payload)
        except Exception as exc:
            if attempt == retries - 1:
                raise
            pause = min(retry_after_seconds(exc, wait * (attempt + 1)), max_wait)
            print(f"  retry {attempt + 1}/{retries - 1} after {type(exc).__name__}, waiting {pause:.0f}s")
            time.sleep(pause)


def load_checkpoint(path: str, id_field: str = "row_id") -> dict:
    """Read an existing JSONL checkpoint into {id: record}."""
    done = {}
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as handle:
            for line in handle:
                line = line.strip()
                if line:
                    record = json.loads(line)
                    done[record[id_field]] = record
    return done


---
# Part 1: Topic Detection and Summarization of News Articles

## Step 1: Load the Dataset

The BBC News Archive is a tab-separated file with 2,225 articles across 5 categories
(business, entertainment, politics, sport, tech).

In [109]:
news_full = pd.read_csv("bbc-news-data.csv", sep="\t")
news_full.insert(0, "Article_ID", news_full.index)
news_full = news_full.rename(columns={"title": "Title", "content": "Article_Text", "category": "True_Category"})

print(news_full.shape)
print(news_full["True_Category"].value_counts())
news_full.head()

(2225, 5)
True_Category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


,Article_ID,True_Category,filename,Title,Article_Text
0,0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [110]:
## Limit to the first 30 articles for the main assignment
news_df = news_full.head(30).copy()
news_df.head()

,Article_ID,True_Category,filename,Title,Article_Text
0,0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


## Step 2: Define the Topic Classification Task (10 marks)

The prompt is few-shot (two short snippets with their correct label) and the output is
constrained with a Pydantic schema, so the model can only return one of the five labels -
that removes the usual "Sure! The topic is Tech." preamble problem.

In [111]:
class TopicLabel(BaseModel):
    """The single news category that best describes the article."""

    topic: Literal["Business", "Entertainment", "Politics", "Sport", "Tech"] = Field(
        description="The one category the article belongs to."
    )


classification_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a news desk editor. Analyze the news article and identify its topic as exactly one "
     "of the following categories: Business, Entertainment, Politics, Sport, or Tech. "
     "Answer with the category label only.\n\n"
     "Examples:\n"
     "Article: \"Shares in the airline fell 4% after it warned that fuel costs would wipe out "
     "half of its annual profit.\" -> Business\n"
     "Article: \"The band announced a 12-date arena tour after their album topped the charts for "
     "a third week.\" -> Entertainment\n"
     "Article: \"The home secretary defended the new immigration bill during a heated Commons "
     "debate.\" -> Politics"),
    ("human", "Title: {title}\n\nArticle: {article}\n\nCategory:"),
])

classification_chain = classification_prompt | llm.with_structured_output(TopicLabel, method="json_schema")

In [112]:
## Sample datapoint
sample_article = news_df.iloc[0]
print("TITLE:", sample_article["Title"])
print("TRUE CATEGORY:", sample_article["True_Category"])

sample_topic = invoke_with_retry(
    classification_chain,
    {"title": sample_article["Title"], "article": sample_article["Article_Text"]},
)
print("PREDICTED TOPIC:", sample_topic.topic)

TITLE: Ad sales boost Time Warner profit
TRUE CATEGORY: business
PREDICTED TOPIC: Business


## Step 3: Define the Summarization Task (10 marks)

A plain string output is enough here, so the chain ends in a `StrOutputParser`. The prompt
explicitly asks for who/what/when/where/why and forbids commentary, which keeps the summaries
factual and roughly the same length across articles.

In [113]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a news summarizer. Summarize the main points of the news article in 2-3 sentences. "
     "Cover who, what, when, where and why where the article states them. "
     "Use only information present in the article, add no personal commentary or opinion, and "
     "return the summary text only."),
    ("human", "Title: {title}\n\nArticle: {article}\n\nSummary:"),
])

summarization_chain = summary_prompt | llm | StrOutputParser()

In [114]:
## Sample datapoint
sample_summary = invoke_with_retry(
    summarization_chain,
    {"title": sample_article["Title"], "article": sample_article["Article_Text"]},
)
print(sample_summary)

US media giant Time Warner reported a 76% rise in quarterly profit to $1.13 billion for the three months ended December, with fourth‑quarter sales up 2% to $11.1 billion, driven by higher high‑speed internet and advertising revenues and one‑off gains that offset a dip at Warner Bros and a loss of AOL subscribers. The company, now holding an 8% stake in Google, also noted an 8% increase in AOL’s underlying profit despite losing 464,000 subscribers, and announced plans to restate its 2000 and 2003 results as the SEC probe concludes, having offered a $300 million settlement. For the full year, Time Warner posted a 27% profit increase to $3.36 billion on revenue of $42.09 billion and projected about 5% operating‑earnings growth in 2005.


## Step 4: Key Entity Extraction (10 marks)

Entities are split into people / organizations / locations so the output is genuinely
structured, and a flat `all_entities` list is also produced for the `Key_Entities` column
(matching the JSON example in the brief).

In [115]:
class KeyEntities(BaseModel):
    """Important named entities mentioned in the article."""

    people: List[str] = Field(default_factory=list, description="Notable people named in the article.")
    organizations: List[str] = Field(default_factory=list, description="Companies, institutions or groups named.")
    locations: List[str] = Field(default_factory=list, description="Countries, cities or places named.")


entity_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You extract named entities from news articles. From the article, list the names of any "
     "important people, organizations and places mentioned. "
     "Only list entities that actually appear in the article; if a group has none, return an "
     "empty list. Do not invent entities and do not repeat the same entity twice."),
    ("human", "Title: {title}\n\nArticle: {article}"),
])

entity_chain = entity_prompt | llm.with_structured_output(KeyEntities, method="json_schema")

In [116]:
## Sample datapoint
sample_entities = invoke_with_retry(
    entity_chain,
    {"title": sample_article["Title"], "article": sample_article["Article_Text"]},
)
print("People       :", sample_entities.people)
print("Organizations:", sample_entities.organizations)
print("Locations    :", sample_entities.locations)

People       : ['Richard Parsons']
Organizations: ['TimeWarner', 'Google', 'AOL', 'Warner Bros', 'US Securities Exchange Commission', 'Bertelsmann', 'AOL Europe']
Locations    : ['US']


In [117]:
def flatten_entities(entities: KeyEntities) -> list:
    """Merge the three entity groups into one de-duplicated list, order preserved."""
    merged = []
    for name in entities.people + entities.organizations + entities.locations:
        name = name.strip()
        if name and name not in merged:
            merged.append(name)
    return merged


flatten_entities(sample_entities)

['Richard Parsons',
 'TimeWarner',
 'Google',
 'AOL',
 'Warner Bros',
 'US Securities Exchange Commission',
 'Bertelsmann',
 'AOL Europe',
 'US']

## Step 5: Update the DataFrame with Results (15 marks)

Each article goes through the three chains in turn. Results are collected into a separate
results frame first (the "new columns" view) and then merged back onto the original data.

In [118]:
def analyze_article(title: str, article: str) -> dict:
    """Run classification, summarization and entity extraction for one article."""
    payload = {"title": title, "article": article}
    topic = invoke_with_retry(classification_chain, payload)
    summary = invoke_with_retry(summarization_chain, payload)
    entities = invoke_with_retry(entity_chain, payload)
    return {
        "Detected_Topic": topic.topic,
        "Summary": summary.strip(),
        "Key_Entities": flatten_entities(entities),
        "People": entities.people,
        "Organizations": entities.organizations,
        "Locations": entities.locations,
    }

In [119]:
## Each finished article is appended to a JSONL checkpoint, and a re-run skips the IDs already
## there. A rate limit part-way through then costs only the rows that are left, not the whole run.
NEWS_CHECKPOINT = "part1_news_checkpoint.jsonl"

news_done = load_checkpoint(NEWS_CHECKPOINT, "Article_ID")
print(f"{len(news_done)}/{len(news_df)} articles already in {NEWS_CHECKPOINT}")

with open(NEWS_CHECKPOINT, "a", encoding="utf-8") as handle:
    for row in news_df.itertuples(index=False):
        if row.Article_ID in news_done:
            continue
        print(f"[{row.Article_ID:>3}] {row.Title[:70]}")
        record = {"Article_ID": int(row.Article_ID), **analyze_article(row.Title, row.Article_Text)}
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        news_done[record["Article_ID"]] = record

news_results_df = pd.DataFrame([news_done[i] for i in news_df["Article_ID"]])[
    ["Article_ID", "Detected_Topic", "Summary", "Key_Entities", "People", "Organizations", "Locations"]
]
print(news_results_df.shape)


30/30 articles already in part1_news_checkpoint.jsonl
(30, 7)


In [120]:
## The new columns on their own
news_results_df[["Article_ID", "Detected_Topic", "Summary", "Key_Entities"]].head(10)

,Article_ID,Detected_Topic,Summary,Key_Entities
0,0,Business,US media giant Time Warner reported a 76% rise...,"[Richard Parsons, TimeWarner, Google, AOL, War..."
1,1,Business,After Federal Reserve Chairman Alan Greenspan ...,"[Alan Greenspan, Robert Sinche, Federal Reserv..."
2,2,Business,"Menatep Group, the owner of the embattled Russ...","[Jamie Firestone, Tim Osborne, Mikhail Khodork..."
3,3,Business,British Airways said that for the three months...,"[Rod Eddington, Mike Powell, Martin Broughton,..."
4,4,Business,Allied Domecq’s London shares rose about 4% af...,"[Allied Domecq, Pernod Ricard, Wall Street Jou..."
5,5,Business,Japan’s economy almost slipped into a technica...,"[Heizo Takenaka, Paul Sheard, Lehman Brothers,..."
6,6,Business,The U.S. Labor Department reported that firms ...,"[President Bush, Herbert Hoover, Rick Egelton,..."
7,7,Business,India’s finance minister Palaniappan Chidambar...,"[Palaniappan Chidambaram, Gordon Brown, G7, Un..."
8,8,Business,A joint FAO‑WFP report says Ethiopia produced ...,"[Henri Josserand, Food and Agriculture Organis..."
9,9,Politics,In 1999 the Clinton administration filed a $28...,"[US government, Clinton administration, Altria..."


In [121]:
## Final merged dataframe: all original columns + all new columns
news_final_df = news_df.merge(news_results_df, on="Article_ID", how="left")
news_final_df.head()

,Article_ID,True_Category,filename,Title,Article_Text,Detected_Topic,Summary,Key_Entities,People,Organizations,Locations
0,0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...,Business,US media giant Time Warner reported a 76% rise...,"[Richard Parsons, TimeWarner, Google, AOL, War...",[Richard Parsons],"[TimeWarner, Google, AOL, Warner Bros, US Secu...",[US]
1,1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...,Business,After Federal Reserve Chairman Alan Greenspan ...,"[Alan Greenspan, Robert Sinche, Federal Reserv...","[Alan Greenspan, Robert Sinche]","[Federal Reserve, Bank of America, G7, White H...","[New York, London, China, Beijing, United Stat..."
2,2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...,Business,"Menatep Group, the owner of the embattled Russ...","[Jamie Firestone, Tim Osborne, Mikhail Khodork...","[Jamie Firestone, Tim Osborne, Mikhail Khodork...","[Yukos, Rosneft, Menatep Group, Reuters, Yugansk]","[Russia, Moscow, US]"
3,3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...,Business,British Airways said that for the three months...,"[Rod Eddington, Mike Powell, Martin Broughton,...","[Rod Eddington, Mike Powell, Martin Broughton,...","[British Airways, Dresdner Kleinwort Wasserste...",[United States]
4,4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...,Business,Allied Domecq’s London shares rose about 4% af...,"[Allied Domecq, Pernod Ricard, Wall Street Jou...",[],"[Allied Domecq, Pernod Ricard, Wall Street Jou...","[UK, France, London, Paris, Scotland, US]"


In [122]:
news_final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Article_ID      30 non-null     int64 
 1   True_Category   30 non-null     str   
 2   filename        30 non-null     str   
 3   Title           30 non-null     str   
 4   Article_Text    30 non-null     str   
 5   Detected_Topic  30 non-null     str   
 6   Summary         30 non-null     str   
 7   Key_Entities    30 non-null     object
 8   People          30 non-null     object
 9   Organizations   30 non-null     object
 10  Locations       30 non-null     object
dtypes: int64(1), object(4), str(6)
memory usage: 2.7+ KB


### Sanity check: detected topic vs the dataset's own label

The dataset ships with a ground-truth category, so the classification step can be scored
directly rather than only eyeballed.

In [123]:
news_final_df["Topic_Correct"] = (
    news_final_df["Detected_Topic"].str.lower() == news_final_df["True_Category"].str.lower()
)
accuracy = news_final_df["Topic_Correct"].mean()
print(f"Topic classification accuracy on {len(news_final_df)} articles: {accuracy:.1%}")

news_final_df.loc[~news_final_df["Topic_Correct"], ["Article_ID", "Title", "True_Category", "Detected_Topic"]]

Topic classification accuracy on 30 articles: 93.3%


,Article_ID,Title,True_Category,Detected_Topic
9,9,Court rejects $280bn tobacco case,business,Politics
14,14,Air passengers win new EU rights,business,Politics


### Example output in JSON

In [124]:
example = news_final_df.iloc[0]
print(json.dumps({
    "Article_ID": int(example["Article_ID"]),
    "Title": example["Title"],
    "Article_Text": example["Article_Text"][:200].strip() + "... [excerpt]",
    "Detected_Topic": example["Detected_Topic"],
    "Summary": example["Summary"],
    "Key_Entities": example["Key_Entities"],
}, indent=2))

{
  "Article_ID": 0,
  "Title": "Ad sales boost Time Warner profit",
  "Article_Text": "Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (\u00a3600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google,... [excerpt]",
  "Detected_Topic": "Business",
  "Summary": "US media giant Time Warner reported a 76% rise in quarterly profit to $1.13\u202fbillion for the three months ended December, driven by higher sales of high\u2011speed internet connections, increased advertising revenue and one\u2011off gains, while fourth\u2011quarter sales grew 2% to $11.1\u202fbillion. The company, which now holds an 8% stake in Google, noted mixed results for its AOL unit\u2014losing 464,000 subscribers but seeing an 8% rise in underlying profit\u2014and a 27% drop in film\u2011division profit due to box\u2011office flops, while full\u2011year profit rose 27% to $3.36\u202fbillion and revenue to $42.09\u202fbillion. T

In [125]:
news_final_df.to_csv("part1_news_analysis_30.csv", index=False)
print("Saved part1_news_analysis_30.csv")

Saved part1_news_analysis_30.csv


---
# Part 2: Job Postings Analysis - Role Categorization and Requirements Extraction

## Step 1: Load the Dataset

In [126]:
jobs_full = pd.read_csv("job_title_des.csv", index_col=0)
jobs_full = jobs_full.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"})
jobs_full = jobs_full.dropna(subset=["Job_Title", "Job_Description"]).reset_index(drop=True)
jobs_full.insert(0, "Job_ID", jobs_full.index)

print(jobs_full.shape)
jobs_full.head()

(2277, 3)


,Job_ID,Job_Title,Job_Description
0,0,Flutter Developer,We are looking for hire experts flutter develo...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ..."
3,3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...
4,4,Full Stack Developer,job responsibility full stack engineer – react...


In [127]:
## Limit to the first 25 job postings for the main assignment
jobs_df = jobs_full.head(25).copy()
jobs_df[["Job_ID", "Job_Title"]]

,Job_ID,Job_Title
0,0,Flutter Developer
1,1,Django Developer
2,2,Machine Learning
3,3,iOS Developer
4,4,Full Stack Developer
5,5,Java Developer
6,6,Full Stack Developer
7,7,JavaScript Developer
8,8,DevOps Engineer
9,9,Software Engineer


## Step 2: Define the Job Category Classification Task (10 marks)

The category list from the brief is extended with the domains that actually show up in this
dataset (Engineering, Operations, HR, Sales, Design), with `Others` as the fallback so the
model never has to force a bad fit.

In [128]:
JOB_CATEGORIES = [
    "Technology/IT",
    "Finance",
    "Marketing",
    "Healthcare",
    "Education",
    "Sales",
    "Human Resources",
    "Design",
    "Engineering",
    "Operations",
    "Others",
]


class JobCategory(BaseModel):
    """The broad domain a job posting belongs to."""

    category: Literal[
        "Technology/IT", "Finance", "Marketing", "Healthcare", "Education",
        "Sales", "Human Resources", "Design", "Engineering", "Operations", "Others",
    ] = Field(description="The single best-fitting domain for this job posting.")


job_category_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a recruitment analyst. Given a job title and description, categorize the job into "
     "exactly one of these domains: " + ", ".join(JOB_CATEGORIES) + ". "
     "Base the decision mainly on the job title, using the description to break ties. "
     "If nothing fits, answer Others. Answer with the domain label only.\n\n"
     "Examples:\n"
     "Job: Django Developer. Description: Strong Python experience in API development... "
     "-> Technology/IT\n"
     "Job: Accounts Executive. Description: Handle GST filings, ledgers and monthly closing... "
     "-> Finance\n"
     "Job: Staff Nurse. Description: Patient care in the ICU ward, administering medication... "
     "-> Healthcare"),
    ("human", "Job: {job_title}.\nDescription: {job_description}\n\nDomain category:"),
])

job_category_chain = job_category_prompt | llm.with_structured_output(JobCategory, method="json_schema")

In [129]:
## Sample datapoint
sample_job = jobs_df.iloc[0]
print("JOB TITLE:", sample_job["Job_Title"])
print("DESCRIPTION:", sample_job["Job_Description"][:300], "...")

sample_category = invoke_with_retry(
    job_category_chain,
    {"job_title": sample_job["Job_Title"], "job_description": sample_job["Job_Description"]},
)
print("\nPREDICTED CATEGORY:", sample_category.category)

JOB TITLE: Flutter Developer
DESCRIPTION: We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.
Job Types: Full-time, Part-time
Salary: ₹20,000.00 - ₹40,000.00 per month
Benefits:
Flexible schedule
Food allowance
Schedule:
Day shift
Supplemental Pay:
Joining bonus
Overtime pay
Ex ...

PREDICTED CATEGORY: Technology/IT


## Step 3: Define the Requirements Extraction Task (30 marks)

Skills, education and experience are pulled in a single composite prompt returning
structured JSON. 

In [130]:
class JobRequirements(BaseModel):
    """Key requirements extracted from a job description."""

    skills: List[str] = Field(
        default_factory=list,
        description="Specific skills, programming languages, tools or domain knowledge required.",
    )
    education: str = Field(
        default="Not specified",
        description="Minimum education level required or preferred, e.g. 'Bachelor's degree in Computer Science'. "
                    "Use 'Not specified' if the description does not state one.",
    )
    experience: str = Field(
        default="Not specified",
        description="Minimum years or level of experience required, e.g. '3+ years' or 'Senior-level'. "
                    "Use 'Not specified' if the description does not state one.",
    )


requirements_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You extract structured requirements from job descriptions. From the description, extract:\n"
     "1. skills - the key skills, technologies, tools or domain knowledge mentioned "
     "(short noun phrases, e.g. 'Python', 'REST APIs', 'project management').\n"
     "2. education - the minimum education level required or preferred.\n"
     "3. experience - the minimum years of experience or experience level required.\n\n"
     "Use only what the description states. Do not infer requirements that are not written down. "
     "If the description does not mention education or experience, return exactly 'Not specified' "
     "for that field. If no skills are mentioned, return an empty list."),
    ("human", "Job: {job_title}.\nDescription: {job_description}"),
])

requirements_chain = requirements_prompt | llm.with_structured_output(JobRequirements, method="json_schema")

In [131]:
## Sample datapoint
sample_requirements = invoke_with_retry(
    requirements_chain,
    {"job_title": sample_job["Job_Title"], "job_description": sample_job["Job_Description"]},
)
print("Skills    :", sample_requirements.skills)
print("Education :", sample_requirements.education)
print("Experience:", sample_requirements.experience)

Skills    : ['Flutter']
Education : Not specified
Experience: 1 year (Preferred)


In [132]:
NOT_SPECIFIED = "Not specified"

def normalize_field(value) -> str:
    """Fold blanks, 'none', 'n/a' and nulls into a single 'Not specified' value."""
    if value is None:
        return NOT_SPECIFIED
    text = str(value).strip()
    if not text or text.lower() in {"none", "n/a", "na", "null", "not mentioned", "not stated", "not specified"}:
        return NOT_SPECIFIED
    return text

## Step 4: Apply the LLM Chain to Each Job Posting (10 marks)

In [133]:
def analyze_job(job_title: str, job_description: str) -> dict:
    """Classify one posting and extract its skills, education and experience."""
    payload = {"job_title": job_title, "job_description": job_description}
    category = invoke_with_retry(job_category_chain, payload)
    requirements = invoke_with_retry(requirements_chain, payload)
    skills = [s.strip() for s in requirements.skills if s and s.strip()]
    return {
        "Predicted_Category": category.category,
        "Required_Skills": skills,
        "Education_Required": normalize_field(requirements.education),
        "Experience_Required": normalize_field(requirements.experience),
    }

In [134]:
JOB_CHECKPOINT = "part2_job_checkpoint.jsonl"

jobs_done = load_checkpoint(JOB_CHECKPOINT, "Job_ID")
print(f"{len(jobs_done)}/{len(jobs_df)} postings already in {JOB_CHECKPOINT}")

with open(JOB_CHECKPOINT, "a", encoding="utf-8") as handle:
    for row in jobs_df.itertuples(index=False):
        if row.Job_ID in jobs_done:
            continue
        print(f"[{row.Job_ID:>3}] {row.Job_Title[:70]}")
        record = {"Job_ID": int(row.Job_ID), **analyze_job(row.Job_Title, row.Job_Description)}
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        jobs_done[record["Job_ID"]] = record

job_results_df = pd.DataFrame([jobs_done[i] for i in jobs_df["Job_ID"]])[
    ["Job_ID", "Predicted_Category", "Required_Skills", "Education_Required", "Experience_Required"]
]
job_results_df


25/25 postings already in part2_job_checkpoint.jsonl


,Job_ID,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,0,Technology/IT,[Flutter],Not specified,1 year (Preferred)
1,1,Technology/IT,"[Python, Django, Flask, REST APIs, RPC, Linux,...",Not specified,Not specified
2,2,Technology/IT,"[Machine Learning, Deep Learning, Python, Java...","Graduate or M.Sc. in Computer Science, Mathema...",3+ years
3,3,Technology/IT,"[iOS development, Objective-C, Cocoa Touch, Co...",Not specified,Not specified
4,4,Technology/IT,"[React, React Native, Redux, Angular, Vue, Jav...",Bachelor's degree in Computer Science or equiv...,"5+ years web development, 2+ years recent Reac..."
5,5,Technology/IT,"[Web services (WSDL, SOAP, RESTful), Relationa...","Bachelor's Degree in Computer Science, Informa...",2 years
6,6,Technology/IT,"[Node.js, Java, MongoDB, Elasticsearch, Redis,...",B.Sc. degree in Computer Science or Engineering,2 years
7,7,Technology/IT,"[ReactJS, NodeJS, Azure Functions, GraphQL, HT...","Any graduation, Any PG, Any Doctorate",3-8 years
8,8,Technology/IT,"[Bash, Ruby, Python, Java, Puppet, Chef, Cloud...",Not specified,Senior-level
9,9,Technology/IT,"[REST API, C/C++, Linux/Unix, Python, Go, Git,...","Minimum of BS or MS; computer engineering, com...",Minimum 7 years of software development experi...


## Step 5: Update the DataFrame with New Columns (5 marks)

In [135]:
## Final merged dataframe: all original columns + all new columns
jobs_final_df = jobs_df.merge(job_results_df, on="Job_ID", how="left")
jobs_final_df.head()

,Job_ID,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,0,Flutter Developer,We are looking for hire experts flutter develo...,Technology/IT,[Flutter],Not specified,1 year (Preferred)
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,Technology/IT,"[Python, Django, Flask, REST APIs, RPC, Linux,...",Not specified,Not specified
2,2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ...",Technology/IT,"[Machine Learning, Deep Learning, Python, Java...","Graduate or M.Sc. in Computer Science, Mathema...",3+ years
3,3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...,Technology/IT,"[iOS development, Objective-C, Cocoa Touch, Co...",Not specified,Not specified
4,4,Full Stack Developer,job responsibility full stack engineer – react...,Technology/IT,"[React, React Native, Redux, Angular, Vue, Jav...",Bachelor's degree in Computer Science or equiv...,"5+ years web development, 2+ years recent Reac..."


In [136]:
jobs_final_df["Predicted_Category"].value_counts()

Predicted_Category
Technology/IT    25
Name: count, dtype: int64

### Spot-check

Three postings are printed in full against their extracted fields, so the output can be read
back against the source text.

In [137]:
for idx in [0, 7, 18]:
    row = jobs_final_df.iloc[idx]
    print("=" * 100)
    print("JOB TITLE :", row["Job_Title"])
    print("CATEGORY  :", row["Predicted_Category"])
    print("SKILLS    :", ", ".join(row["Required_Skills"]) or "Not specified")
    print("EDUCATION :", row["Education_Required"])
    print("EXPERIENCE:", row["Experience_Required"])
    print("-" * 100)
    print("DESCRIPTION:\n", row["Job_Description"][:900], "...")
    print()

JOB TITLE : Flutter Developer
CATEGORY  : Technology/IT
SKILLS    : Flutter
EDUCATION : Not specified
EXPERIENCE: 1 year (Preferred)
----------------------------------------------------------------------------------------------------
DESCRIPTION:
 We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.
Job Types: Full-time, Part-time
Salary: ₹20,000.00 - ₹40,000.00 per month
Benefits:
Flexible schedule
Food allowance
Schedule:
Day shift
Supplemental Pay:
Joining bonus
Overtime pay
Experience:
total work: 1 year (Preferred)
Housing rent subsidy:
Yes
Industry:
Software Development
Work Remotely:
Temporarily due to COVID-19 ...

JOB TITLE : JavaScript Developer
CATEGORY  : Technology/IT
SKILLS    : ReactJS, NodeJS, Azure Functions, GraphQL, HTML5, CSS3, JavaScript, REST, written communication, verbal communication
EDUCATION : Any graduation, Any PG, Any Doctorate
EXPERIENCE: 3-8 years
--------------------------------------------------------

### Example output in JSON

In [138]:
example_job = jobs_final_df.iloc[0]
print(json.dumps({
    "Job_Title": example_job["Job_Title"],
    "Job_Description": example_job["Job_Description"][:250].strip() + "... [excerpt]",
    "Predicted_Category": example_job["Predicted_Category"],
    "Required_Skills": example_job["Required_Skills"],
    "Education_Required": example_job["Education_Required"],
    "Experience_Required": example_job["Experience_Required"],
}, indent=2))

{
  "Job_Title": "Flutter Developer",
  "Job_Description": "We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.\r\nJob Types: Full-time, Part-time\r\nSalary: \u20b920,000.00 - \u20b940,000.00 per month\r\nBenefits:\r\nFlexible schedule\r\nFood allowance\r\nSchedule:\r\nDay shift... [excerpt]",
  "Predicted_Category": "Technology/IT",
  "Required_Skills": [
    "Flutter"
  ],
  "Education_Required": "Not specified",
  "Experience_Required": "1 year (Preferred)"
}


In [139]:
jobs_final_df.to_csv("part2_job_analysis_25.csv", index=False)
print("Saved part2_job_analysis_25.csv")

Saved part2_job_analysis_25.csv


---
# Bonus (optional) - Running on the full datasets

The brief warns that Groq will rate-limit a full-dataset run and recommends a local SLM via
Ollama instead, so the bonus uses `llama3.2` (3B) served locally. Three changes make the full
run practical:

1. **One call per row.** Instead of three chains per article, a single structured call returns
   topic, summary and entities together. That cuts a 2,225-article run from ~6,675 calls to 2,225.
2. **Checkpointing.** Every row is appended to a JSONL file as it completes, and the runner skips
   IDs already present. A crash, a rate limit or a closed laptop costs one row, not the whole run.
3. **Truncation.** Article and description text is capped before being sent, since a 3B model
   with a long context is where most of the wall-clock time goes.

The cells below are written to be run directly. Both runs are long (the news run is roughly an
hour or two on CPU; the job dataset has ~63k postings, so `JOB_LIMIT` is set to a slice you can
raise as far as you are willing to let it run). Set `RUN_BONUS = True` to start.

In [140]:
RUN_BONUS = True  ## flip to False to skip the full-dataset bonus

NEWS_LIMIT = len(news_full)   ## all 2,225 articles
JOB_LIMIT = 500               ## raise towards len(jobs_full) (~63k) as time allows

In [141]:
from langchain_ollama import ChatOllama

## Pull the model first (one-off):  ollama pull llama3.2
ollama_llm = ChatOllama(model="llama3.2", temperature=0)

In [142]:
## The Part 1 and Part 2 prompts are reused unchanged - only the model swaps to the local one.
## The prompts that must return a fixed label or a list keep their structured-output wrapper,
## since a 3B model's free text is not reliably parseable.
MAX_CHARS = 4000


def truncate_text(text, max_chars=MAX_CHARS):
    """Cap article/description length - long context is where a 3B model on CPU spends its time."""
    return ("" if text is None else str(text))[:max_chars]


classification_chain_ollama = classification_prompt | ollama_llm.with_structured_output(TopicLabel)
summary_chain_ollama = summary_prompt | ollama_llm | StrOutputParser()
entity_chain_ollama = entity_prompt | ollama_llm.with_structured_output(KeyEntities)
job_category_chain_ollama = job_category_prompt | ollama_llm.with_structured_output(JobCategory)
requirements_chain_ollama = requirements_prompt | ollama_llm.with_structured_output(JobRequirements)

In [ ]:
## Process ALL BBC News articles using Ollama
df_news_all = news_full.head(NEWS_LIMIT).copy().reset_index(drop=True)
total = len(df_news_all)
print(f"Processing all {total} BBC News articles with Ollama...")

all_topics = []
all_summaries = []
all_entities = []

for idx, row in df_news_all.iterrows():
    payload = {"title": row["Title"], "article": truncate_text(row["Article_Text"])}
    if (idx + 1) % 50 == 0 or idx == 0:
        print(f"  Processing {idx + 1}/{total}...")

    try:
        topic = classification_chain_ollama.invoke(payload).topic
        summary = summary_chain_ollama.invoke(payload).strip()
        entities = flatten_entities(entity_chain_ollama.invoke(payload))
    except Exception as exc:
        ## A single bad row should not end a multi-hour run.
        print(f"  row {idx} failed: {type(exc).__name__}: {exc}")
        topic, summary, entities = None, None, []

    all_topics.append(topic)
    all_summaries.append(summary)
    all_entities.append(entities)

df_news_all["Detected_Topic"] = all_topics
df_news_all["Summary"] = all_summaries
df_news_all["Key_Entities"] = all_entities
df_news_all.to_csv("bonus_part1_news_analysis_full.csv", index=False)

accuracy = (df_news_all["Detected_Topic"].str.lower()
            == df_news_all["True_Category"].str.lower()).mean()
print(f"\nDone! Final DataFrame shape: {df_news_all.shape}")
print(f"Rows analyzed: {df_news_all['Detected_Topic'].notna().sum()} / {total}")
print(f"Topic accuracy (llama3.2 3B, full dataset): {accuracy:.1%}")
df_news_all[["Title", "True_Category", "Detected_Topic", "Summary", "Key_Entities"]].head(10)

Processing all 2225 BBC News articles with Ollama...
  Processing 1/2225...


In [ ]:
## Process job postings using Ollama
df_jobs_all = jobs_full.head(JOB_LIMIT).copy().reset_index(drop=True)
total = len(df_jobs_all)
print(f"Processing {total} job postings with Ollama...")

all_categories = []
all_skills = []
all_education = []
all_experience = []

for idx, row in df_jobs_all.iterrows():
    title = row["Job_Title"]
    payload = {"job_title": title, "job_description": truncate_text(row["Job_Description"])}
    if (idx + 1) % 100 == 0 or idx == 0:
        print(f"  Processing {idx + 1}/{total}: {title[:40]}...")

    try:
        category = job_category_chain_ollama.invoke(payload).category
        requirements = requirements_chain_ollama.invoke(payload)
        skills = [s.strip() for s in requirements.skills if s and s.strip()]
        education = normalize_field(requirements.education)
        experience = normalize_field(requirements.experience)
    except Exception as exc:
        print(f"  row {idx} failed: {type(exc).__name__}: {exc}")
        category, skills, education, experience = None, [], NOT_SPECIFIED, NOT_SPECIFIED

    all_categories.append(category)
    all_skills.append(skills)
    all_education.append(education)
    all_experience.append(experience)

df_jobs_all["Predicted_Category"] = all_categories
df_jobs_all["Required_Skills"] = all_skills
df_jobs_all["Education_Required"] = all_education
df_jobs_all["Experience_Required"] = all_experience
df_jobs_all.to_csv("bonus_part2_job_analysis_full.csv", index=False)

print(f"\nDone! Final DataFrame shape: {df_jobs_all.shape}")
print(f"Rows analyzed: {df_jobs_all['Predicted_Category'].notna().sum()} / {total}")
display(df_jobs_all["Predicted_Category"].value_counts())
df_jobs_all[["Job_Title", "Predicted_Category", "Required_Skills", "Education_Required", "Experience_Required"]].head(10)